# Classification by a neural network using PyTorch -- Penguins Classification

## 0. Import packages and modules

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns

import sklearn
import torch

print(sklearn.__version__)
print(torch.__version__)

In [ ]:
# print GPU info

if torch.cuda.is_available():
    print(f'Using GPU: {torch.cuda.get_device_name(torch.cuda.current_device())}')
    device = torch.device('cuda')
else:
    print(f'Using CPU')
    device = torch.device('cpu')

## 1. Formulate/outline the problem: penguin classification

## 2. Identify inputs and outputs

In [ ]:
penguins = sns.load_dataset('penguins')
penguins.head()

In [ ]:
penguins.shape

In [ ]:
# sns.pairplot(penguins.iloc[:, 1:8], hue="species") # `1:8` means without the first (rowid) and the last column (year)

sns.pairplot(
    penguins[["species", "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]],
    hue="species",
    height=2.5
)

## 3. Prepare data

In [ ]:
# drop categorical columns

penguins_filtered = penguins.drop(columns=['island', 'sex'])
penguins_filtered.head(7)

In [ ]:
# drop rows that have NaN values

penguins_filtered = penguins_filtered.dropna()
penguins_filtered.head(7)

In [ ]:
# Extract columns corresponding to features
features = penguins_filtered.drop(columns=['species'])
features

In [ ]:
target = pd.get_dummies(penguins_filtered['species'])
target.head(5)   # print out the top 5 to see what it looks like.

In [ ]:
target.sample(7) # randomly pickup 7 examples from the dataset

**📝 Exercise: One-hot encoding**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=0, shuffle=True, stratify=target
)

In [ ]:
# define and fit a scaler

from sklearn.preprocessing import RobustScaler

feature_scaler = RobustScaler()
X_train_scaled = feature_scaler.fit_transform(X_train)
X_test_scaled = feature_scaler.transform(X_test)

## 4. Build an architecture from scratch

In [ ]:
# set two random seeds, one for numpy and one for tensorflow

from numpy.random import seed
seed(1)

torch.manual_seed(2)

**TODO:** Resolve all the `FIXME` comments below

In [ ]:
import torch.functional as F

# NOTE: use torch.nn.Linear for all layers, and
# remember to pass the previous shape as first parameter

class PenguinModel(torch.nn.Module):
    def __init__(self, input_shape):
        super().__init__()
        # NOTE: there is no "input" layer in PyTorch
        
        # FIXME: define
        # - self.hidden_layer with 10 neurons
        # - self.output_layer with 3 neurons

    def forward(self, x):
        x = self.hidden_layer(x)
        # FIXME: use activation function F.relu(x)
        x = self.output_layer(x)
        # FIXME: use activation function F.softmax(x, dim=1)
        return x

model = PenguinModel(X_train.shape[1]).to(device)

In [ ]:
# Alternative syntax for quickly defining a simple model

# model = torch.nn.Sequential(
#     torch.nn.Linear(X_train.shape[1], 10),
#     torch.nn.ReLU(),
#     torch.nn.Linear(10, 3),
#     torch.nn.Softmax(dim=1)
# ).to(device)

In [ ]:
from torchinfo import summary

summary(model, input_size = (X_train.shape[1:]), batch_dim = 0, device = device)

**📝 Exercise: Create the neural network**

## 5. Choose a loss function and optimizer

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

## 6. Train model

In [ ]:
train_dataset = torch.utils.data.TensorDataset(
    torch.tensor(X_train_scaled, dtype = torch.float),
    torch.tensor(y_train.values, dtype = torch.float)
)
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size = 128, shuffle = True)

In [ ]:
model.train()
history = {
    'loss': []
}
epochs = 100

for epoch in range(epochs):
    running_loss = 0.0

    for X_batch, y_batch in train_dataloader:
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        running_loss += loss.item()

    train_loss = running_loss / len(train_dataloader)
    history['loss'].append(train_loss)
    print(f'Epoch {epoch+1:>3d} completed; loss: {train_loss:.4f}')

In [ ]:
sns.lineplot(x=range(epochs), y=history['loss'])

**📝 Exercise: The Training Curve**

## 7. Perform a prediction/classification

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(torch.tensor(X_test_scaled, dtype = torch.float, device = device))

prediction = pd.DataFrame(y_pred.to('cpu'), columns=target.columns)
prediction

In [ ]:
# idxmax will select the column for each row with the highest value
predicted_species = prediction.idxmax(axis="columns")
predicted_species

## 8. Measuring performance

In [ ]:
from sklearn.metrics import confusion_matrix

true_species = y_test.idxmax(axis="columns")

matrix = confusion_matrix(true_species, predicted_species)
print(matrix)

In [ ]:
# Convert to a pandas dataframe
confusion_df = pd.DataFrame(matrix, index=y_test.columns.values, columns=y_test.columns.values)

# Set the names of the x and y axis, this helps with the readability of the heatmap.
confusion_df.index.name = 'True Label'
confusion_df.columns.name = 'Predicted Label'
confusion_df.head()

In [ ]:
sns.heatmap(confusion_df, annot=True)

## 9. Refine the model

## 10. Share model

In [ ]:
import joblib

joblib.dump(feature_scaler, 'penguins_scaler.gz')
torch.save(model.state_dict(), 'penguins_classification.pt')

In [ ]:
pretrained_model = PenguinModel(X_train.shape[1]).to(device)
pretrained_model.load_state_dict(torch.load('penguins_classification.pt'))
pretrained_scaler = joblib.load('penguins_scaler.gz')

In [ ]:
# use the pretrained model here
model.eval()
with torch.no_grad():
    X_test_scaled = torch.tensor(pretrained_scaler.transform(X_test), dtype = torch.float, device = device)
    y_pretrained_pred = model(X_test_scaled)
pretrained_prediction = pd.DataFrame(y_pretrained_pred.to('cpu'), columns=target.columns.values)

# idxmax will select the column for each row with the highest value
pretrained_predicted_species = pretrained_prediction.idxmax(axis="columns")
print(pretrained_predicted_species)